# plotmux — comparing backends

This notebook renders the same `plotmux` calls with every registered backend side by side, to compare their output for the same backend-agnostic spec.

In [ ]:
import numpy as np

import plotmux

rng = np.random.default_rng(42)
values = rng.normal(loc=0.0, scale=1.0, size=100_000)
x = np.linspace(0.0, 10.0, 200)
y = np.sin(x)
scatter_x = rng.uniform(0.0, 10.0, size=200)
scatter_y = scatter_x + rng.normal(scale=0.5, size=200)

BACKENDS = ["matplotlib", "xy"]

## Basic histogram

Render the same histogram with each backend and display them one after the other.

In [ ]:
for name in BACKENDS:
    fig = plotmux.hist(values, bins=101, backend=name)
    display(fig.to_native())

## Custom bin count, axis range, and density

The same backend-agnostic arguments (`bins`, `xmin`, `xmax`, `density`) produce equivalent histograms across backends.

In [ ]:
for name in BACKENDS:
    fig = plotmux.hist(values, bins=101, xmin="q0.01", xmax="q0.99", density=True, backend=name)
    display(fig.to_native())

## Label

`label` is a backend-agnostic argument, forwarded to each backend's native naming/legend mechanism.

In [ ]:
for name in BACKENDS:
    fig = plotmux.hist(values, bins=101, density=True, label="group A", backend=name)
    display(fig.to_native())

## Color

Colors are normalized once via `plotmux.colors.parse_color`, so hex strings, named colors, and RGB(A) tuples all render consistently across backends.

In [ ]:
for name in BACKENDS:
    fig = plotmux.hist(values, bins=101, color="#ff8800", backend=name)
    display(fig.to_native())

## Titles, axis labels, and log scale

`title`, `xlabel`, `ylabel`, and `yscale="log"` should look equivalent across backends.

In [ ]:
for name in BACKENDS:
    fig = plotmux.hist(
        values,
        bins=101,
        title="Standard normal sample",
        xlabel="value",
        ylabel="count",
        yscale="log",
        backend=name,
    )
    display(fig.to_native())

## Line chart

Compare `plotmux.line` across backends.

In [ ]:
for name in BACKENDS:
    fig = plotmux.line(x, y, label="sin(x)", color="tab:blue", backend=name)
    display(fig.to_native())

## Scatter chart

Compare `plotmux.scatter`, including the `size` argument, across backends.

In [ ]:
for name in BACKENDS:
    fig = plotmux.scatter(
        scatter_x, scatter_y, label="noisy y = x", color="tab:red", size=20, backend=name
    )
    display(fig.to_native())

## Layering multiple specs

Compare `plotmux.layer` (scatter + fitted line on shared axes) across backends.

In [ ]:
coeffs = np.polyfit(scatter_x, scatter_y, deg=1)
fit_x = np.linspace(scatter_x.min(), scatter_x.max(), 50)
fit_y = np.polyval(coeffs, fit_x)

for name in BACKENDS:
    fig = plotmux.layer(
        plotmux.scatter(scatter_x, scatter_y, label="data", color="tab:gray"),
        plotmux.line(fit_x, fit_y, label="fit", color="tab:red"),
        title="Linear fit",
        backend=name,
    )
    display(fig.to_native())

## Saving a figure

`Figure.save` infers the export format from the file suffix and delegates to the backend.

In [ ]:
for name in BACKENDS:
    fig = plotmux.hist(values, bins=101, backend=name)
    fig.save(f"../tmp/plotmux_hist_{name}.svg")